## Bonus Assignment - Rotary Positional Embeddings (RoPE)

###  **Objective**

Implement the **Rotary Positional Embedding (RoPE)** mechanism from scratch. You will:

1.  Precompute the frequency tensors ($\cos$ and $\sin$) for a given sequence length.
2.  Implement the `rotate_half` and `apply_rope` functions to efficiently rotate query and key vectors.
3.  Integrate this into a `RoPEMultiHeadAttention` layer and verify that the attention mechanism now understands relative positions without any added absolute positional embeddings.

### **Assignment Structure**

This assignment will contain:

1.  **Setup Block:** Imports, configuration, and the standard `MultiHeadAttention` class (from Week 1) to serve as a base.
2.  **Part 1: Frequency Precomputation:** Calculating the geometric progression of angles.
3.  **Part 2: The Rotation Logic:** Implementing the efficient rotation trick.
4.  **Part 3: `RoPEMultiHeadAttention`:** Putting it all together.

In [8]:
import torch
import torch.nn as nn
import math

# --- Configuration ---
BATCH_SIZE = 2
SEQ_LEN = 10
D_MODEL = 32
NUM_HEADS = 4
HEAD_DIM = D_MODEL // NUM_HEADS # 8

# --- Reproducibility ---
torch.manual_seed(42)

print("="*50)
print(f"Config: HEAD_DIM={HEAD_DIM}")
print("="*50)

Config: HEAD_DIM=8


-----

### **Part 1: Frequency Precomputation**

**Instructions:** RoPE relies on precomputing rotation angles based on the position index $m$ and the dimension index $i$.
The formula for the frequency $\theta_i$ at dimension $i \in [0, d/2)$ is:
$$\theta_i = 10000^{-2i/d}$$
You need to generate the `cos` and `sin` tables for all positions up to `max_len`.

In [9]:
print("\n--- Part 1: Frequency Precomputation ---")

def precompute_freqs_cis(dim, max_len=1000, theta=10000.0):
    """
    Precomputes the cosine and sine values for RoPE.

    Args:
        dim (int): The dimension of the head (HEAD_DIM).
        max_len (int): Maximum sequence length.
        theta (float): The base frequency constant.

    Returns:
        freqs_cos (torch.Tensor): Shape (1, 1, max_len, dim)
        freqs_sin (torch.Tensor): Shape (1, 1, max_len, dim)
    """
    # 1. Calculate the frequencies for the half-dimension (dim // 2)
    # formula: theta ^ (-2i / dim)
    # Hint: use torch.arange(0, dim, 2)
    freqs = torch.arange(0, dim, 2).float()
    freqs = theta ** (freqs / dim)

    # 2. Create position indices (0 to max_len)
    t = torch.arange(max_len).float()

    # 3. Calculate the outer product of positions (t) and frequencies
    # Shape should be (max_len, dim // 2)
    freqs = torch.outer(t, freqs)

    # 4. RoPE applies the same rotation to pairs of dimensions.
    # We need to concatenate the frequencies with themselves to match
    # the full 'dim' shape.
    # e.g. [freq0, freq1] -> [freq0, freq1, freq0, freq1]
    freqs = torch.cat((freqs, freqs), dim=-1)

    # 5. Calculate cos and sin
    freqs_cos = torch.cos(freqs)
    freqs_sin = torch.sin(freqs)

    # 6. Reshape for broadcasting: (1, 1, max_len, dim)
    # This allows us to apply it to (Batch, Num_Heads, Seq_Len, Dim)
    return freqs_cos.view(1, 1, max_len, dim), freqs_sin.view(1, 1, max_len, dim)

# --- Verification for Part 1 ---
print("--- Running Verification for Part 1 ---")
try:
    cos_test, sin_test = precompute_freqs_cis(dim=HEAD_DIM, max_len=SEQ_LEN)
    assert cos_test.shape == (1, 1, SEQ_LEN, HEAD_DIM)
    assert sin_test.shape == (1, 1, SEQ_LEN, HEAD_DIM)
    # Check that cos^2 + sin^2 = 1
    assert torch.allclose(cos_test**2 + sin_test**2, torch.ones_like(cos_test))
    print(" (1a) Shapes and trig identities are correct.")
    print("Part 1 seems correct!")
except Exception as e:
    print(f" Part 1 Failed: {e}")


--- Part 1: Frequency Precomputation ---
--- Running Verification for Part 1 ---
 (1a) Shapes and trig identities are correct.
Part 1 seems correct!


#### **Questions for Part 1**

**Q1: Conceptual Check**
Why do we calculate frequencies for only `dim // 2` dimensions and then concatenate them?

  * (A) Because RoPE rotates vectors in 2D sub-spaces (pairs of dimensions).
  * (B) To save memory.
  * (C) Because the second half of the embedding vector is always zero.
  * (D) Because gradients only flow through the first half.


**Q2: Value Check**
Run `precompute_freqs_cis` with `dim=32` and `max_len=10`. What is the value of `freqs_cos[0, 0, 5, 2]` (Position 5, Dimension index 2)?

In [10]:
# Code for Q2
cos_q2, _ = precompute_freqs_cis(32, 10)
print(f"Q2 Value: {cos_q2[0, 0, 5, 2].item()}")

Q2 Value: -0.9946564435958862




-----

### **Part 2: The Rotation Logic**

**Instructions:** Implement the actual rotation.

1.  `rotate_half(x)`: Swaps the first half of the vector with the second half and negates the first half. Input $[x_1, x_2]$, Output $[-x_2, x_1]$.
2.  `apply_rope(x, cos, sin)`: Computes $x_{new} = x \cdot \cos + \text{rotate_half}(x) \cdot \sin$.

In [11]:
print("\n--- Part 2: The Rotation Logic ---")

def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    # x shape: (..., dim)
    # Split x into two halves x1, x2
    x1, x2 = x.chunk(2, dim=-1)
    # Return concatenated tensor: [-x2, x1]
    return torch.cat((-x2, x1), dim=-1)

def apply_rope(x, cos, sin):
    """
    Applies Rotary Position Embedding.

    Args:
        x: Query or Key tensor (Batch, Heads, Seq, Dim)
        cos: Precomputed Cosine (1, 1, Max_Len, Dim)
        sin: Precomputed Sine (1, 1, Max_Len, Dim)

    Returns:
        x_rotated
    """
    # 1. Slice cos and sin to the sequence length of x
    # (x.size(2) is the sequence length)
    seq_len = x.size(2)
    cos = cos[:, :, :seq_len, :]
    sin = sin[:, :, :seq_len, :]

    # 2. Apply the RoPE formula:
    # (x * cos) + (rotate_half(x) * sin)
    # Note: This implements the rotation matrix multiplication element-wise
    x_rotated = x * cos + rotate_half(x) * sin

    return x_rotated

# --- Verification for Part 2 ---
print("--- Running Verification for Part 2 ---")
try:
    torch.manual_seed(42)
    x_dummy = torch.randn(1, 1, 2, 4) # 2 positions, 4 dim
    cos_dummy, sin_dummy = precompute_freqs_cis(4, 2)

    out_dummy = apply_rope(x_dummy, cos_dummy, sin_dummy)
    assert out_dummy.shape == x_dummy.shape
    print(" (2a) Output shape is correct.")

    # Manual check for pos 0 (rotation by 0 radians -> Identity)
    # cos(0)=1, sin(0)=0. Result should be x.
    assert torch.allclose(out_dummy[:,:,0,:], x_dummy[:,:,0,:])
    print(" (2b) Position 0 rotation is Identity (Correct).")
    print(" Part 2 seems correct!")
except Exception as e:
    print(f" Part 2 Failed: {e}")


--- Part 2: The Rotation Logic ---
--- Running Verification for Part 2 ---
 (2a) Output shape is correct.
 (2b) Position 0 rotation is Identity (Correct).
 Part 2 seems correct!


#### **Questions for Part 2**

**Q3: Value Check**
Create a vector `x` of shape `(1, 1, 1, 4)` with values `[1.0, 2.0, 3.0, 4.0]`. Run `rotate_half(x)`. What is the **sum** of the resulting tensor?

In [12]:
# Code for Q3
x_q3 = torch.tensor([1.0, 2.0, 3.0, 4.0]).view(1, 1, 1, 4)
out_q3 = rotate_half(x_q3)
print(out_q3)
print(f"Q3 Sum: {out_q3.sum().item()}")

tensor([[[[-3., -4.,  1.,  2.]]]])
Q3 Sum: -4.0




**Q4: Value Check**
Using `torch.manual_seed(42)`, generate `x` (shape 1,1,5,8) and compute `apply_rope`. What is the **sum** of the final rotated tensor?

In [13]:
# Code for Q4
torch.manual_seed(42)
x_q4 = torch.randn(1, 1, 5, 8)
cos_q4, sin_q4 = precompute_freqs_cis(8, 5)
out_q4 = apply_rope(x_q4, cos_q4, sin_q4)
print(f"Q4 Rotated Sum: {out_q4.sum().item()}")

Q4 Rotated Sum: -9.33022689819336




-----

### **Part 3: `RoPEMultiHeadAttention`**

**Instructions:** Create a new MHA class. It should be almost identical to your Week 1 MHA, but:

1.  In `__init__`, call `precompute_freqs_cis`.
2.  In `forward`, apply `apply_rope` to `Q` and `K` **after** reshaping them to multi-head view, but **before** the dot product.
3.  **Important:** Do NOT use any `PositionalEncoding` module from Week 2. RoPE handles it.


In [16]:
print("\n--- Part 3: RoPEMultiHeadAttention ---")

class RoPEMultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, max_len=1000):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)

        # 1. Precompute and register buffers
        cos, sin = precompute_freqs_cis(self.head_dim, max_len)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        seq_len = q.size(1)

        # 1. Projections
        Q = self.wq(q).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.wk(k).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.wv(v).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # 2. Apply RoPE to Q and K
        # Note: apply_rope expects shape (Batch, Heads, Seq, Dim) which matches Q and K here
        Q_rope = apply_rope(Q, self.cos, self.sin)
        K_rope = apply_rope(K, self.cos, self.sin)

        # 3. Scaled Dot-Product Attention
        # (Standard implementation)
        d_k = self.head_dim
        scores = torch.matmul(Q_rope, K_rope.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = torch.softmax(scores, dim=-1)
        context = torch.matmul(attn, V)

        # 4. Concatenate and Output
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        output = self.wo(context)
        return output

# --- Verification for Part 3 ---
print("--- Running Verification for Part 3 ---")
try:
    torch.manual_seed(42)
    rope_mha = RoPEMultiHeadAttention(D_MODEL, NUM_HEADS, max_len=SEQ_LEN)
    x_in = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL)
    output = rope_mha(x_in, x_in, x_in)

    assert output.shape == x_in.shape
    print(" (3a) Output shape is correct.")
    print(" Part 3 seems correct!")
except Exception as e:
    print(f" Part 3 Failed: {e}")


--- Part 3: RoPEMultiHeadAttention ---
--- Running Verification for Part 3 ---
 (3a) Output shape is correct.
 Part 3 seems correct!


#### **Questions for Part 3**

**Q5: Conceptual Check**
Unlike the original Transformer, where absolute PE is added to the embeddings, RoPE is applied:

  * (A) To the values (V) only.
  * (B) To the query (Q) and key (K) vectors at every layer.
  * (C) To the input embeddings before the first layer only.
  * (D) To the attention scores (softmax output).



**Q6: Value Check (The Ultimate Check)**
Initialize `RoPEMultiHeadAttention(d_model=32, num_heads=4)` with `seed=42`.
Generate input `x = torch.randn(2, 10, 32)` with `seed=42`.
Run the forward pass. What is the **mean** of the output tensor?

In [17]:
# Code for Q6
torch.manual_seed(42)
mha_q6 = RoPEMultiHeadAttention(32, 4, max_len=10)
x_q6 = torch.randn(2, 10, 32)
out_q6 = mha_q6(x_q6, x_q6, x_q6)
print(f"Q6 Output Mean: {out_q6.mean().item()}")

Q6 Output Mean: 0.003695417195558548




**Q7: Value Check (Relative Distance Invariance)**
This checks the core property of RoPE: the dot product $Q_m \cdot K_n$ should depend primarily on $m-n$.
We will compute attention scores for two pairs with the same relative distance.

1.  Pair A: Query at pos 5, Key at pos 3 (dist=2).
2.  Pair B: Query at pos 8, Key at pos 6 (dist=2).
    Calculate the difference between their raw dot products (scores). Ideally, it should be small (though not exactly zero due to the randomness of the vectors, RoPE guarantees the *rotation* is relative).
    Actually, to test the implementation precisely, calculate the **dot product** of the rotated Q (pos 2) and K (pos 0) for a specific head.

<!-- end list -->

In [18]:
# Code for Q7
torch.manual_seed(42)
# Create a single head dimension vector
q_vec = torch.ones(1, 1, 1, 8) # "Query"
k_vec = torch.ones(1, 1, 1, 8) # "Key"
cos, sin = precompute_freqs_cis(8, 10)

# Rotate Q as if it's at position 2
q_rot = apply_rope(q_vec, cos[:,:,2:3,:], sin[:,:,2:3,:])
# Rotate K as if it's at position 0
k_rot = apply_rope(k_vec, cos[:,:,0:1,:], sin[:,:,0:1,:])

# Dot product
dot_prod = (q_rot * k_rot).sum()
print(f"Q7 Dot Product: {dot_prod.item()}")

Q7 Dot Product: 0.2233266830444336




-----